# indicators_001

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (165).ipynb`

| Field | Value |
|---|---|
| Section | `indicators` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 19 |
| Detected functions | generate_signals, process_trade |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Copy of merged_file_10ind.parquet')
data

In [ ]:
data.columns

In [ ]:
import pandas as pd

def generate_signals(data):
    """
    Generate trading signals assuming 'Probability' is already in the dataset.

    Parameters:
    - data (pd.DataFrame): Dataset with 'Close', 'Kalman', 'VWAP', 'ADX', and 'Probability'.

    Returns:
    - data (pd.DataFrame): Updated DataFrame with 'Signal' column.
    """
    # Compute Proposed_Signal
    data['Proposed_Signal'] = 0
    long_condition = (data['close'] > data['Kalman_Close']) & (data['Kalman_Close'] > data['VWAP']) & (data['ADX'] > 30)
    short_condition = (data['close'] < data['Kalman_Close']) & (data['Kalman_Close'] < data['VWAP']) & (data['ADX'] > 30)
    data.loc[long_condition, 'Proposed_Signal'] = 1
    data.loc[short_condition, 'Proposed_Signal'] = -1

    return data

# Example usage
# data = pd.read_csv('your_data_with_probability.csv')
data = generate_signals(data)

In [ ]:
data.set_index('Date', inplace=True)

In [ ]:
data.drop('Signal', axis=1)

In [ ]:
!pip install xgboost dask

In [ ]:
import pandas as pd
import numpy as np

# Parameters
INITIAL_CAPITAL = 100_000
BROKERAGE_PER_TRADE = 101
SLIPPAGE_PERCENTAGE = 0.0001
ATR_MULTIPLIER_TP = 15
ATR_MULTIPLIER_SL = 10

# State Tracking
tradebook = []
current_positions = {}
cumulative_pnl = 0

# Convert DataFrame to NumPy array for faster iteration
data_np = data[['Ticker', 'Proposed_Signal', 'open', 'low', 'high', 'close', 'ATR', 'date']].reset_index().to_numpy()

def process_trade(idx, datetime, ticker, signal, price, low, high, close, atr):
    """Processes trades based on entry, exit, and stop-loss conditions."""
    global current_positions, cumulative_pnl

    if pd.isnull(price) or price <= 0 or pd.isnull(atr) or atr <= 0:
        print(f"Skipping invalid data for {ticker} at {datetime}")
        return

    trade_time = pd.Timestamp(datetime).time()

    # Handle existing position
    if ticker in current_positions:
        entry_price, trade_type, entry_time = current_positions[ticker]
        position_size = INITIAL_CAPITAL / entry_price

        tp = entry_price + atr * ATR_MULTIPLIER_TP if trade_type == 'long' else entry_price - atr * ATR_MULTIPLIER_TP
        sl = entry_price - atr * ATR_MULTIPLIER_SL if trade_type == 'long' else entry_price + atr * ATR_MULTIPLIER_SL

        exit_price, exit_reason = None, None

        # Check TP/SL hits
        if (trade_type == 'long' and (low <= sl or high >= tp)) or (trade_type == 'short' and (high >= sl or low <= tp)):
            exit_price = tp if (trade_type == 'long' and high >= tp) or (trade_type == 'short' and low <= tp) else sl
            exit_price *= (1 - SLIPPAGE_PERCENTAGE) if trade_type == 'long' else (1 + SLIPPAGE_PERCENTAGE)
            exit_reason = 'TP Hit' if exit_price == tp else 'SL Hit'

        # Force exit at 15:28 (end of day)
        if trade_time >= pd.Timestamp("15:28").time():
            exit_price = close * (1 - SLIPPAGE_PERCENTAGE) if trade_type == 'long' else close * (1 + SLIPPAGE_PERCENTAGE)
            exit_reason = 'EOD Close'

        if exit_price is not None:
            pnl = (exit_price - entry_price) * position_size if trade_type == 'long' else (entry_price - exit_price) * position_size
            pnl -= 2 * BROKERAGE_PER_TRADE
            cumulative_pnl += pnl

            tradebook.append([
                ticker, entry_time, datetime, entry_price, exit_price,
                trade_type, pnl, cumulative_pnl, exit_reason
            ])
            del current_positions[ticker]

    # Handle new signal (only before 15:15)
    if ticker not in current_positions and signal in [1, -1] and trade_time <= pd.Timestamp("15:15").time():
        trade_type = 'long' if signal == 1 else 'short'
        entry_price = price * (1 + SLIPPAGE_PERCENTAGE) if trade_type == 'long' else price * (1 - SLIPPAGE_PERCENTAGE)
        current_positions[ticker] = (entry_price, trade_type, datetime)
        print(f"New {trade_type} position opened for {ticker} at {datetime} with entry price {entry_price:.2f}")

# Process each row
for idx, (datetime, ticker, signal, price, low, high, close, atr, date) in enumerate(data_np):
    process_trade(idx, datetime, ticker, signal, price, low, high, close, atr)

# Convert tradebook to DataFrame
tradebook_df = pd.DataFrame(
    tradebook,
    columns=['Ticker', 'Entry Time', 'Exit Time', 'Entry Price', 'Exit Price', 'Trade Type', 'PnL', 'Cumulative PnL', 'Exit Reason']
)

print(tradebook_df)

# Let me know if you want to add more features or performance metrics! 🚀

In [ ]:
r = tradebook_df.to_csv('/content/drive/MyDrive/tradebook_10.csv')

In [ ]:
data.dropna()

In [ ]:
data

In [ ]:

# Step 1: Merge the datasets
'''
data['date'] = pd.to_datetime(data['date'])
tradebook_df['Entry Time'] = pd.to_datetime(tradebook_df['Entry Time'])

# Rename 'date' to 'Entry Time' in data
data.rename(columns={'date': 'Entry Time'}, inplace=True)
'''
# Merge, keeping all columns from 'data' but only rows with matching trades
merged_df = data.merge(tradebook_df, on=['Ticker', 'Entry Time'], how='left')

# Create the 'Profitable' column
merged_df['Profitable'] = (merged_df['PnL'] > 0).astype(int)

In [ ]:
merged_df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Ensure 'merged_df' is preloaded with all necessary features and the target label
# merged_df should be your dataset with features + 'Profitable' column as the target

# Step 2: Prepare features and labels
features = merged_df[['open', 'high', 'low', 'close', 'volume',
                     'Kalman_Close', 'TR', 'DM+', 'DM-', 'DI+', 'DI-', 'DX',
                     'ADX', 'cum_volume', 'cum_vwap', 'VWAP', 'VWAP_20', 'RSI',
                     'EMA12', 'EMA26', 'MACD', 'Signal', 'ATR', 'hour', 'Proposed_Signal',
                     'minute', 'day', '5min_change', '10min_change', '15min_change']]

labels = merged_df['Profitable']

# Step 3: Handle NaN and Inf values
# Convert infinite values to NaN
features.replace([np.inf, -np.inf], np.nan, inplace=True)

# Fill NaN values with column mean
features.fillna(0, inplace=True)

# Step 4: Train-test split
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# Step 5: Scale the features (helps prevent large-value bias)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Step 6: Train the XGBoost model
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# Step 7: Make predictions and filter trades
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Only keep trades with probability > 0.6
high_prob_trades = X_test[y_pred_proba > 0.6]  # Filter based on confidence level
high_prob_trades = pd.DataFrame(high_prob_trades, columns=features.columns)  # Convert back to DataFrame
high_prob_trades['Predicted_Prob'] = y_pred_proba[y_pred_proba > 0.6]

# Step 8: Evaluate the model
accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Model Accuracy: {accuracy * 100:.2f}%")
print(classification_report(y_test, model.predict(X_test)))

# Step 9: Display high-confidence trades
print("High Probability Trades:")
print(high_prob_trades)


In [ ]:
high_prob_trades[0:50]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
feature_importances = model.feature_importances_
feature_names = features.columns

# Get the top 10 important features
sorted_idx = np.argsort(feature_importances)[-10:]

plt.barh(feature_names[sorted_idx], feature_importances[sorted_idx])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance')
plt.show()
# Let me know if

In [ ]:
x = feature_names[sorted_idx]
print(x)

In [ ]:
high_prob_trades.to_csv('/content/drive/MyDrive/high_prob_trades.csv')